#개요
Olist 데이터셋 가지고 text-to-sql FineTuning용 데이터셋 만들기.

#환경설정

In [2]:
!pip install langchain-openai langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.4 MB/s eta 0:00:00


In [111]:
import json
import re
import pandas as pd
import random
import sqlite3
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [222]:
os.environ["OPENAI_API_KEY"] = ""

In [207]:
llm = ChatOpenAI(model = "gpt-5", temperature = 0.3)
llm_sql = ChatOpenAI(model = 'gpt-4o-mini', temperature=0)

목표

instruction : "DDL Statements:DDL문\n입력:한글프롬프트"

input : ""

output : SQL 문

In [28]:
with open('/content/gretelai_text_to_sql_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

[
  {
    "instruction": "DDL statements:\nCREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');\n입력 텍스트: 각 판매원이 판매한 목재의 총량은 얼마이며, 판매원에 따라 정렬되어 있나요?\n\n위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.",
    "input": "",
    "output": "쿼리 작성: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;"
  },
  {
    "instruction": "DDL statements:\nCREATE TABLE equipment_maintenance (equipment_type VARCHAR(255), maintenance_frequency INT);\n입력 텍스트: 장비 유형과 해당 장비의 전체 유지 보

데이터셋 로드

In [6]:
df_customers        = pd.read_csv('olist_customers_dataset.csv')
df_geolocation      = pd.read_csv('olist_geolocation_dataset.csv')
df_order_items      = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments   = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews    = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders           = pd.read_csv('olist_orders_dataset.csv')
df_products         = pd.read_csv('olist_products_dataset.csv')
df_sellers          = pd.read_csv('olist_sellers_dataset.csv')
df_product_category = pd.read_csv('product_category_name_translation.csv')

# 유용한 함수

In [218]:
def parse_llm_output(llm_output: str, num_pairs: int) -> list[dict]:
    """
    LLM 출력을 파싱해 질문-SQL 딕셔너리를 담은 리스트로 반환

    Args:
        llm_output : LLM 출력 문자열
        num_pairs  : 파싱할 질문-SQL 쌍의 개수

    Returns:
        [{'Question': '...', 'SQL': '...'}, ...]
    """
    questions = re.findall(r'\[질문\]\s*(.*?)\s*(?=\[SQL\])', llm_output, re.DOTALL)
    sqls      = re.findall(r'\[SQL\]\s*(.*?)\s*(?=\[질문\]|$)',  llm_output, re.DOTALL)

    # 백틱 제거
    sqls = [re.sub(r'```sql|```', '', sql).strip() for sql in sqls]
    questions = [q.strip() for q in questions]

    result = []
    for i in range(min(num_pairs, len(questions), len(sqls))):
        result.append({
            'Question': questions[i],
            'SQL':      sqls[i]
        })

    return result

In [219]:
def convert_to_sqlite(sql: str) -> str:
    """
    DB 종류에 따라 다른 SQL문을 LLM을 사용해 SQLite 문법으로 변환

    Args:
        sql : 변환할 SQL문

    Returns:
        SQLite 문법으로 변환된 SQL문
    """
    response = llm_sql.invoke(
        f"""다음 SQL문을 SQLite 문법으로 변환해줘.
반드시 SQL문만 출력하고 다른 설명은 절대 추가하지 마.
백틱이나 코드블록 없이 순수 SQL문만 출력해.

{sql}"""
    )
    return response.content.strip()


def execute_sql_on_db(parsed_results: list[dict], conn) -> list[dict]:
    """
    질문-SQL 딕셔너리를 담은 List를 받아 DB에 전체 실행해보기.
    """

    results = []

    for item in parsed_results:
        question  = item['Question']
        sql       = item['SQL'].rstrip(';')
        sql_sqlite = None

        # SQL 유형 판별
        sql_type = sql.strip().split()[0].upper()  # SELECT, UPDATE, DELETE, INSERT

        def run_sql(query):
            if sql_type == 'SELECT':
                # SELECT는 pd.read_sql_query() 사용
                return pd.read_sql_query(query, conn), 'success'
            else:
                # INSERT, UPDATE, DELETE는 cursor로 실행 후 롤백
                cursor = conn.cursor()
                cursor.execute(query)
                affected = cursor.rowcount
                conn.rollback()  # 실제 반영 안 되게 롤백
                return pd.DataFrame({'affected_rows': [affected]}), 'success'

        # 1차 시도: 원본 SQL 실행
        try:
            df_result, status = run_sql(sql)
        except Exception as e:
            df_result = None
            status    = f'error: {e}'

            # 2차 시도: LLM으로 SQLite 변환 후 재실행
            print(f"❌ 1차 실행 실패 → LLM으로 SQLite 변환 시도")
            try:
                sql_sqlite        = convert_to_sqlite(sql).rstrip(';')
                df_result, status = run_sql(sql_sqlite)
            except Exception as e2:
                df_result = None
                status    = f'error (변환 후에도 실패): {e2}'

        results.append({
            'Question'  : question,
            'SQL'       : sql,
            'SQL_SQLite': sql_sqlite,
            'Result'    : df_result,
            'Status'    : status
        })

        print(f"Question:   {question}")
        print(f"SQL 원본:   {sql}")
        if sql_sqlite:
            print(f"SQL 변환:   {sql_sqlite}")
        print(f"Status:     {status}")
        print("\n")
        print(df_result if df_result is not None else "")
        print("\n")
        print("-" * 50)

    return results

In [220]:
def get_sample_values(df, n=3) -> str:
  """
  dataframe의 각 칼럼 별 unique한 값 중 랜덤으로 n개를 뽑은 결과 반환

  인수 :
    dataframe : olist 데이터프레임
    n : 랜덤으로 추출할 값 개수
  return :
    dataframe 각 칼럼에서 unique 한 값 중 랜덤으로 n개를 뽑은 결과 str
  """
  sample_text = ""
  for col in df.columns:
      uniques = df[col].dropna().unique().tolist()
      samples = random.sample(uniques, min(n, len(uniques)))  # 랜덤으로 n개 추출
      sample_text += f"{col} : {samples}\n"
  return sample_text

In [217]:
def convert_to_gretel_format(total_result: list[dict], ddl: str) -> list[dict]:
    """
    total_results를 gretelai text-to-sql 학습 데이터 형식으로 변환

    Args:
        total_results : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
        ddl           : instruction에 넣을 DDL 문자열

    Returns:
        [{'instruction': ..., 'input': '', 'output': ...}, ...]
    """
    converted = []
    for item in total_result:
        question = item['Question']
        sql      = item['SQL']

        instruction = (
            f"DDL statements:\n{ddl}\n"
            f"입력 텍스트: {question}\n\n"
            f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
        )

        converted.append({
            "instruction": instruction,
            "input"      : "",
            "output"     : f"쿼리 작성: {sql}"
        })

    return converted

#1.단일 SQL 문 생성

데이터 8개 * 50개 = 400행


In [203]:
SYSTEM_PROMPT = """
#역할
당신은 text-to-SQL을 수행해야 합니다.
DDL 선언문, 칼럼 설명, 칼럼별 값 예시를 참고해 사용자의 입장에서 할 수 있는 자연어 질문과 그에 대응하는 SQL문을 작성해주세요.
질문-SQL 쌍을 3개 생성하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 아래 SELECT, INSERT, UPDATE, DELETE SQL 패턴을 고르게 사용할 수 있는 질문-sql 쌍을 만드세요.
   [SELECT]
   - 단순 조회 (SELECT, WHERE)
   - 집계 (COUNT, AVG, SUM, MIN, MAX + GROUP BY)
   - 정렬 및 제한 (ORDER BY, LIMIT)
   - NULL 처리 (IS NULL, IS NOT NULL, COALESCE)
   - 날짜 연산 (날짜 범위 조회, 날짜 차이 계산)
   - 서브쿼리 (IN, EXISTS, 스칼라 서브쿼리)
   - 조건 분기 (CASE WHEN)
   - 중복 제거 (DISTINCT)
   [INSERT]
   - 단일 행 삽입
   - 다중 행 삽입
   [UPDATE]
   - 단일 컬럼 수정
   - 조건부 다중 컬럼 수정
   [DELETE]
   - 조건부 행 삭제
4. 칼럼별 값 예시를 활용해 구체적인 값을 포함한 질문을 생성하세요.

#주의
1. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.

2. SQL 작성 시 주의
   기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다.
   기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'

3. 현재 날짜/시간
현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [204]:
HUMAN_PROMPT = """
# DDL Statement
{DDL_Statement}

# 칼럼 설명
{column_descriptions}

# 칼럼별 값 예시
{column_values}

# history
{history}
"""

In [208]:
def generate_sql(DDL_Statement: str, column_descriptions: str, column_values : str,
                 history : list[dict]) -> str:
    """
    DDL문과 컬럼 설명을 입력받아 LLM으로 질문-SQL 쌍을 생성

    Args:
        DDL_Statement       : 테이블 DDL 문
        column_descriptions : 컬럼 설명

    Returns:
        LLM이 생성한 질문-SQL 쌍 문자열
    """
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
        HumanMessagePromptTemplate.from_template(HUMAN_PROMPT),
    ])

    chain = prompt | llm

    # history 중 질문만 str로 취합
    history_str = ""
    for res in total_result:
      history_str += res.get('Question') + "\n"

    response = chain.invoke({
        "DDL_Statement"      : DDL_Statement,
        "column_descriptions": column_descriptions,
        "column_values" : column_values,
        "history" : history_str
    })
    # print(response.usage_metadata) # 캐싱 작동 확인용
    return response.content

## df_orders

In [209]:
# 변수 설정
  #1. DDL 선언문
DDL_Statement = """
CREATE TABLE orders (
    order_id                        VARCHAR(32)     NOT NULL PRIMARY KEY,
    customer_id                     VARCHAR(32)     NOT NULL,
    order_status                    VARCHAR(50)     NOT NULL,
    order_purchase_timestamp        DATETIME        NOT NULL,
    order_approved_at               DATETIME,
    order_delivered_carrier_date    DATETIME,
    order_delivered_customer_date   DATETIME,
    order_estimated_delivery_date   DATETIME
);
"""
  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.
"""

  #3. 실제 DB 값.
column_values = get_sample_values(df_orders)

#LLM 실행
total_result = []
for i in range(3):
  print(f'진행중 : {i+1}')
  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        history = total_result)

  #LLM 응답 파싱 후 저장.
  parsed = parse_llm_output(result, 3)
  total_result.extend(parsed)

진행중 : 0
{'input_tokens': 1310, 'output_tokens': 5414, 'total_tokens': 6724, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4736}}
진행중 : 1
{'input_tokens': 1547, 'output_tokens': 6155, 'total_tokens': 7702, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5376}}
진행중 : 2
{'input_tokens': 1872, 'output_tokens': 5258, 'total_tokens': 7130, 'input_token_details': {'audio': 0, 'cache_read': 1536}, 'output_token_details': {'audio': 0, 'reasoning': 4544}}


In [211]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

❌ 1차 실행 실패 → LLM으로 SQLite 변환 시도
Question:   2018년 5월에 구매된 주문 중 상태가 'delivered'였고 실제 배송이 예정보다 늦은 건만 보고 싶습니다. 주문 ID, 구매 시각, 예상 배송일, 실제 배송일, 지연일수(일) 기준으로 지연이 큰 순서 상위 10건을 알려주세요.
SQL 원본:   SELECT
  order_id,
  order_purchase_timestamp,
  order_estimated_delivery_date,
  order_delivered_customer_date,
  TIMESTAMPDIFF(DAY, order_estimated_delivery_date, order_delivered_customer_date) AS delay_days
FROM orders
WHERE DATE(order_purchase_timestamp) >= '2018-05-01'
  AND DATE(order_purchase_timestamp) < '2018-06-01'
  AND order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
  AND order_delivered_customer_date > order_estimated_delivery_date
ORDER BY delay_days DESC, order_purchase_timestamp DESC
LIMIT 10
SQL 변환:   SELECT
  order_id,
  order_purchase_timestamp,
  order_estimated_delivery_date,
  order_delivered_customer_date,
  julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) AS delay_days
FRO

In [221]:
#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement)

with open('Olist_orders_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

✅ 9개 변환 완료!


#2.JOIN 있는 SQL문 생성

#3.SQL문 실행해보는 법(평가)

In [ ]:
# 메모리에 임시 DB 생성
conn = sqlite3.connect(':memory:')

# DataFrame을 DB 테이블로 올리기
df_orders.to_sql('df_orders', conn, if_exists='replace', index=False)

# SQL 실행
result = pd.read_sql_query("""
    SELECT * FROM df_orders LIMIT 5
""", conn)

In [ ]:
result

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
